In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json

In [2]:
def get_products_for_rate_bases(products_path, rate_bases_path):
    with open(products_path, 'r') as f:
        products = json.load(f)
    
    with open(rate_bases_path, 'r') as f:
        rate_bases = json.load(f)

    valid_rate_base_codes = set(rb.get("code") for rb in rate_bases if "code" in rb)

    filtered_products = [
        p for p in products
        if p.get("rateBasis") in valid_rate_base_codes
    ]

    return filtered_products


def append_to_json_file(data, file):
    if os.path.exists(file):
        return 
    
    with open(file, 'w', encoding='utf-8') as json_file:
        json.dump(data, json_file, indent=4, ensure_ascii=False)
    print(f"Data saved to {file}")

In [4]:
def mergeUnitsAndCalendar(units_path, calendar_path):
    """
    Make the units data and unit calendar data into one data based on unit.
    """
    with open(units_path, 'r') as f:
        units_data = json.load(f)
    
    with open(calendar_path, 'r') as f:
        calendar_data = json.load(f)
    
    units_lookup = {
        (item['unit'], item['code']): {
            'contracted': item['contracted'],
            'bases': item['bases'],
            'createdOn': item['createdOn']
        }
        for item in units_data
    }
    
    merged_data = []
    for calendar_item in calendar_data:
        key = (calendar_item['unit'], calendar_item['code'])
        if key in units_lookup:
            merged_item = {
                'unit': calendar_item['unit'],
                'code': calendar_item['code'],
                'contracted': units_lookup[key]['contracted'],
                'bases': units_lookup[key]['bases'],
                'createdOn': units_lookup[key]['createdOn'],
                'calendar': calendar_item.get('calendar', [])
            }
            merged_data.append(merged_item)
    
    return merged_data

In [5]:
calendar_data = mergeUnitsAndCalendar('../Data/ExtractedData/allotmentsUnits.json', '../Data/ExtractedData/allotmentsCalendar.json')
append_to_json_file(calendar_data, '../Data/MainData/allotmentsData.json') 

Data saved to ../Data/MainData/allotmentsData.json


In [6]:
def merged_products_Calendar(prducts_path, calendar_path):
    """
    Add the details of the products to the calendarProducts data.
    """

    with open(prducts_path, 'r') as f:
        products = json.load(f)
    
    with open(calendar_path, 'r') as f:
        calendar = json.load(f)

    products_lookup = {
        (item['unit'], item['code']): {
            'board': item['board'],
            'rateBasis': item['rateBasis'],
            'occupancy': item['occupancy'],
            'currency': item['currency'],
            'ratePlanCode': item['ratePlanCode'],
            'beds': item['beds'],
            'createdOn': item['createdOn']
        } for item in products if 'unit' in item and 'code' in item
    }

    merged_data = []
    for cal_item in calendar:
        key = (cal_item['unit'], cal_item['code'])
        if key in products_lookup:
            merged_item = {
                'unit': cal_item['unit'],
                'code': cal_item['code'],
                'board': products_lookup[key]['board'],
                'rateBasis': products_lookup[key]['rateBasis'],
                'occupancy': products_lookup[key]['occupancy'],
                'currency': products_lookup[key]['currency'],
                'ratePlanCode': products_lookup[key]['ratePlanCode'],
                'beds': products_lookup[key]['beds'],
                'createdOn': products_lookup[key]['createdOn'],
                'calendar': cal_item.get('calendar', [])
            }
            merged_data.append(merged_item)
        else:
            print(f"Key not found: {key}")
    
    return merged_data

In [7]:
merged_product_data = merged_products_Calendar('../Data/ExtractedData/products.json', '../Data/ExtractedData/productsCalendar.json')
append_to_json_file(merged_product_data, '../Data/MainData/productsData.json')

Data saved to ../Data/MainData/productsData.json
